In [ ]:
"""
Regulatory Affairs RAG AI Assistant
"""

# Standard library
import os
import shutil
import tempfile
from typing import Dict, List

# Third-party
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# LangChain core
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# LangChain integrations
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain retrievers / loaders
from langchain_classic.document_loaders import WebBaseLoader
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.document_loaders import UnstructuredXMLLoader


load_dotenv()

True

In [7]:
# ============================================================
# Data Models
# ============================================================
class ResearchResponse(BaseModel):
    """Structured response from the research assistant."""

    answer: str = Field(description="The answer to the question")
    confidence: str = Field(description="high, medium, or low based on source quality")
    sources: List[str] = Field(description="List of source documents used")
    key_quotes: List[str] = Field(
        description="Relevant quotes from sources", default=[]
    )
    follow_up_questions: List[str] = Field(description="Suggested follow-up questions")


In [8]:
# ============================================================
# Database
# ============================================================

class Database:

    def ingest_urls(self, urls: dict[str, str]):

            shutil.rmtree("./research_db", ignore_errors=True)
            documents = []

            for title, url in urls.items():

                if url.endswith(".xml") or "lawdata" in url:
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()

                    with tempfile.NamedTemporaryFile(
                        mode="w",
                        suffix=".xml",
                        delete=False,
                        encoding="utf-8"
                    ) as f:
                        f.write(r.text)
                        temp_xml = f.name

                    try:
                        loader = UnstructuredXMLLoader(temp_xml)
                        docs = loader.load()
                    finally:
                        os.remove(temp_xml)
                else:
                    loader = WebBaseLoader(url)
                    docs = loader.load()

                for d in docs:
                    d.metadata["source"] = url
                    d.metadata["title"] = title

                documents.extend(docs)

            chunks = self.splitter.split_documents(documents)

            self.vectorstore.add_documents(chunks)

            print(f"Ingested {len(documents)} documents")
            print(f"Indexed {len(chunks)} chunks")
            print(f"Total in DB: {self.vectorstore._collection.count()}")

In [9]:
# ============================================================
# Regulatory Affairs RAG AI Assistant
# ============================================================

class AIResearchAssistant:
    """AI Research Assistant with document ingestion and retrieval."""
    
    def ingest_urls(self, urls: dict[str, str]):

            shutil.rmtree("./research_db", ignore_errors=True)
            documents = []

            for title, url in urls.items():

                if url.endswith(".xml") or "lawdata" in url:
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()

                    with tempfile.NamedTemporaryFile(
                        mode="w",
                        suffix=".xml",
                        delete=False,
                        encoding="utf-8"
                    ) as f:
                        f.write(r.text)
                        temp_xml = f.name

                    try:
                        loader = UnstructuredXMLLoader(temp_xml)
                        docs = loader.load()
                    finally:
                        os.remove(temp_xml)
                else:
                    loader = WebBaseLoader(url)
                    docs = loader.load()

                for d in docs:
                    d.metadata["source"] = url
                    d.metadata["title"] = title

                documents.extend(docs)

            chunks = self.splitter.split_documents(documents)

            self.vectorstore.add_documents(chunks)

            print(f"Ingested {len(documents)} documents")
            print(f"Indexed {len(chunks)} chunks")
            print(f"Total in DB: {self.vectorstore._collection.count()}")

    def __init__(
        self,
        persist_directory: str = "./research_db",
        chunk_size: int = 1000,
        chunk_overlap: int = 200,
    ):
        self.persist_directory = persist_directory

        # 1. Embeddings - turns text into vectors
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        # 2. Splitter - breaks big docs into chunks
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

        # 3. Vector store - stores and searches embeddings
        self.vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name="research_docs",
        )

        self.session_store: Dict[str, InMemoryChatMessageHistory] = {}

        print(f"Research Assistant initialized")
        print(f"  Vector store: {persist_directory}")
        print(f"  Documents indexed: {self.vectorstore._collection.count()}")
              
    def _build_retriever(self):

        # Base retrieval
        base_retriever = self.vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k":8,
                "fetch_k":30
            }
        )

        # Multi-query expansion
        stateless_llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        multi_retriever = MultiQueryRetriever.from_llm(
            retriever=base_retriever,
            llm=stateless_llm,
        )

        # Compression layer
        compressor = LLMChainExtractor.from_llm(
            stateless_llm
        )

        compression_retriever = ContextualCompressionRetriever(
            base_retriever=multi_retriever,
            base_compressor=compressor
        )

        return compression_retriever
    
    def _format_docs_for_context(self, docs) -> str:
        """Format retrieved documents into a string for the prompt."""
        if not docs:
            return "No relevant documents found."

        formatted = []
        for i, doc in enumerate(docs):
            source = doc.metadata.get("source", "Unknown")
            formatted.append(f"[Source {i+1}: {source}]\n{doc.page_content}")
        return "\n\n---\n\n".join(formatted)
    
    def _get_session_history(self, session_id: str) -> BaseChatMessageHistory:
        """Get or create session history."""
        if session_id not in self.session_store:
            self.session_store[session_id] = InMemoryChatMessageHistory()
        return self.session_store[session_id]

    def ask_simple(
        self,
        question: str,
        session_id: str = "default",
        ) -> str:
        """Ask a question against the research documents."""

        # Get memory
        history = self._get_session_history(session_id)

        # Retrieve
        retriever = self._build_retriever()
        docs = retriever.invoke(question)
        context = self._format_docs_for_context(docs)
        sources = list(set(d.metadata.get("source", "Unknown") for d in docs))

        # Prompt -- tell the LLM about available sources
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """You are an AI Research Assistant for regulatory compliance for pharmaceutical and medical device industries.
                    Analyze the provided contexta and sources and return a structured response within 200 words.
                    
                    Content: {context}
                    Sources: {sources}

                    Rules:
                    1. ONLY use information from the provided context and chat history
                    2. If the context doesn't have the answer, say so in the answer field
                    3. Set confidence: "high" if directly stated, "medium" if inferred, "low" if partial
                    4. Include the source filenames you actually used
                    5. Extract key quotes word-for-word from the context
                    6. Suggest 2-3 follow-up questions the user might want to ask

                    Use conversation history to understand follow-up questions.""",
                ),
                MessagesPlaceholder(variable_name="history"),
                (
                    "human","Question: {question}",
                ),
            ]
        )

        chain = prompt | self.llm | StrOutputParser()

        simple_response = ""
        for token in chain.stream(
            {
            "context": context,
            "question": question,
            "sources": ", ".join(sources),
            "history": history.messages[-10:]
            }
        ):
            print(token, end="", flush=True)
            simple_response += token

        # Save to memory (store just the answer text)
        history.add_message(HumanMessage(content=question))
        history.add_message(AIMessage(content=simple_response))

        return simple_response
    

    def ask_structured(
            self,
            question: str,
            session_id: str = "default",
        ) -> ResearchResponse:
            """Ask a question and get a structured response."""

            # LLM that returns a Pydantic object instead of a string
            structured_llm = self.llm.with_structured_output(ResearchResponse)

            # Get memory
            history = self._get_session_history(session_id)

            # Retrieve
            retriever = self._build_retriever()
            docs = retriever.invoke(question)
            context = self._format_docs_for_context(docs)
            sources = list(set(d.metadata.get("source", "Unknown") for d in docs))

            # Prompt -- tell the LLM about available sources
            prompt = ChatPromptTemplate.from_messages(
                [
                    (
                        "system",
                        """You are an AI Research Assistant for regulatory compliance for pharmaceutical and medical device industries.
                        Analyze the provided contexta and sources and return a structured response within 200 words.
                        
                        Content: {context}
                        Sources: {sources}

                        You Must follow these Rules:
                        1. ONLY use information from the provided context and chat history
                        2. If the context doesn't have the answer, say so in the answer field
                        3. Set confidence: "high" if directly stated, "medium" if inferred, "low" if partial
                        4. Include the source filenames you actually used
                        5. Extract key quotes word-for-word from the context
                        6. Suggest 2-3 follow-up questions the user might want to ask
                        
                        Use conversation history to understand follow-up questions.""",
                    ),
                    MessagesPlaceholder(variable_name="history"),
                    (
                        "human","Question: {question}",
                    ),
                ]
            )

            chain = prompt | structured_llm

            structured_response = chain.invoke(
                {
                    "context": context,
                    "question": question,
                    "sources": ", ".join(sources),
                    "history": (
                        history.messages[-10:]
                        if hasattr(history, "messages")
                        else history[-10:]
                    ),
                }
            )

            # Save to memory (store just the answer text)
            history.add_message(HumanMessage(content=question))
            history.add_message(AIMessage(content=structured_response.answer))

            return structured_response
        
    def clear_session(self, session_id: str):
        if session_id in self.session_store:
            self.session_store[session_id].clear()
            print(f"Cleared session: {session_id}")

In [10]:
dict_urls = {
        "Pharmaceuticals and Medical Devices Act": "https://laws.e-gov.go.jp/api/1/lawdata/335AC0000000145",
        "EudraLex - EU Legislation": "https://health.ec.europa.eu/medicinal-products/eudralex_en",
        "CFR_Title_21 SUBCHAPTER D—DRUGS FOR HUMAN USE": "https://www.govinfo.gov/content/pkg/CFR-2025-title21-vol5/xml/CFR-2025-title21-vol5-chapI-subchapD.xml"
    }

In [6]:
db = AIResearchAssistant()
db.ingest_urls(dict_urls)

Research Assistant initialized
  Vector store: ./research_db
  Documents indexed: 0
Ingested 3 documents
Indexed 1653 chunks
Total in DB: 1653


In [12]:
questions = ["Hello, I am Hippocrates of Kos from the ancient Greece. Let me ask you some questions about regulatory affairs.",
            "Do you recall my name and location?",
            "Please summarize the structure of 'Pharmaceuticals and Medical Devices Act(Japan)','EudraLex(EU)' and 'CFR_Title_21 SUBCHAPTER D—DRUGS FOR HUMAN USE(United States)' respectively.",
            "Based on the summary of the structure you created ealier, what are the key differences of the structure between those three regulations?"]

assistant = AIResearchAssistant()
assistant.clear_session("default")
for question in questions:
    print(f"Q: {question}\n")
    print("Simple Answer:")
    assistant.ask_simple(question)
    print(f"\n")
    print(f"Structured Answer: \n{assistant.ask_structured(question)}\n\n")

Research Assistant initialized
  Vector store: ./research_db
  Documents indexed: 1653
Q: Hello, I am Hippocrates of Kos from the ancient Greece. Let me ask you some questions about regulatory affairs.

Simple Answer:
Hello Hippocrates! I'm here to assist you with your questions about regulatory affairs in the pharmaceutical and medical device industries. Please go ahead and ask your questions! 

**Follow-up questions you might want to ask:**
1. What are the key steps in the drug approval process?
2. How does the FDA handle adverse drug experiences?
3. What are the responsibilities of medical professionals regarding drug safety?

Structured Answer: 
answer="Hello Hippocrates! I'm here to assist you with your questions about regulatory affairs in the pharmaceutical and medical device industries. Please go ahead and ask your questions!" confidence='high' sources=[] key_quotes=[] follow_up_questions=['What specific aspects of regulatory affairs are you interested in?', 'Are you looking fo